# 03 - Trajectory Visualization

Visualize smartphone GNSS trajectories in a local East-North-Up frame and compare the route categories available in the dataset.

In [ ]:
import sys, os
from pathlib import Path
for p in ("..", "."):
    if (Path(p) / "src").is_dir():
        sys.path.insert(0, os.path.abspath(p)); break

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data.io_vnbd_loader import IOVNBDDataset
from src.data.smartphone_extractor import SmartphoneExtractor
from src.preprocessing.coordinate_transforms import geodetic_to_enu

ds = IOVNBDDataset("data/raw")
ex = SmartphoneExtractor(ds)

## 1. Build a local ENU trajectory for one trip

In [ ]:
def trajectory(tid):
    d = ex.extract_trip(tid).data.dropna(subset=["latitude_deg", "longitude_deg"])
    if not len(d):
        return None, None
    ref = (float(d["latitude_deg"].iloc[0]), float(d["longitude_deg"].iloc[0]))
    e, n, u = geodetic_to_enu(d["latitude_deg"], d["longitude_deg"], 0, *ref)
    return (e, n, u, d["timestamp"].to_numpy()), d

In [ ]:
t, d = trajectory("vw16b")
e, n, u, ts = t
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(e, n, lw=1.0)
ax.set(xlabel="east (m)", ylabel="north (m)", title="Vw16b trajectory (chip-tar/wet road)")
ax.axis("equal")
plt.show()

## 2. Compare route categories

In [ ]:
cats = {"vta": "A-road", "vtb": "A-road variant", "vw": "chip-tar/wet", "vf": "motorway/city"}
fig, ax = plt.subplots(figsize=(8, 8))
cmap = plt.cm.tab10
for k, label in cats.items():
    tids = [t for t in ds.smartphone_trip_ids() if t.startswith(k)][:2]
    for j, tid in enumerate(tids):
        t, dd = trajectory(tid)
        if t is None:
            continue
        e, n, _, _ = t
        col = cmap(list(cats.keys()).index(k) % 10)
        ax.plot(e, n, lw=0.9, color=col, label=label if j == 0 else None)
ax.set(xlabel="east (m)", ylabel="north (m)", title="Trajectories by road category")
ax.axis("equal")
ax.legend(loc="best")
plt.show()

## 3. GNSS fix gaps

In [ ]:
t, d = trajectory("vta10")
g = d[d["latitude_deg"].notna()]
dt = g["timestamp"].diff().dropna()
print("fix interval stats (s):")
print(dt.describe().round(2).to_string())
plt.figure(figsize=(12, 2))
plt.eventplot((g["timestamp"] - g["timestamp"].iloc[0]).to_numpy())
plt.xlabel("fix time since start (s)")
plt.title("GNSS fix availability over time (vta10)")
plt.show()

## 4. Takeaways

- Trajectories are coherent over tens of km; the local ENU frame is a clean base for fusion.
- Categories cover A-roads, chip-tar/wet roads, motorway and city driving - good variety for training and dead-reckoning evaluation.
- GNSS updates are bursty; `synchronize_data.py` forward-fills them onto the 10 Hz IMU grid.